# SegmentEveryForam Workflow: Foraminiferal Segmentation and Morphometric Analysis

# Project Description

This project aims to extract size information from foram images. The process involved segmenting forams, interacting with the segmented object, and extracting morphometric features. 

**Adapted from Segmenteverygrain:** https://github.com/zsylvester/segmenteverygrain

### Import Libraries and load models (We only do this once)

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import keras
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import segmenteveryforam as sef
import segmenteveryforam.interactions as sfi
from tqdm import tqdm
from PIL import Image
import cv2
import numpy as np
from scipy import stats

%matplotlib qt

In [ ]:
#Use this code to see the version of python, numpy (works better on version 1.26.4 or lower)
import sys
import tensorflow as tf
import torch

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Torch:", torch.__version__)
print("SegmentEveryForam:", sef.__version__)

In [ ]:
# ============================================================
# LOAD U-NET AND SAM 2.1 MODELS
# ============================================================

#Change the path below to point to the unet_model and sam2.1 checkpoint. 
from pathlib import Path

# Place the Colab checkpoint in models/ to use the evaluated model.
best_validation = Path("../models/best_val_loss.keras")
shipped_v2 = Path("../models/seg_model_foram_v2_G_ruber_30epochs.keras")
unet_model = best_validation if best_validation.is_file() else shipped_v2
sam_checkpoint = "../models/sam2.1_hiera_large.pt"

# Load U-Net model
unet = keras.saving.load_model(unet_model, custom_objects={"weighted_crossentropy": sef.weighted_crossentropy},)

print("U-Net model loaded.")

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

# Load SAM 2.1
sam = build_sam2("configs/sam2.1/sam2.1_hiera_l.yaml", sam_checkpoint, device=device,)
predictor = SAM2ImagePredictor(sam)

print("SAM 2.1 model loaded.")

# This is where you begin for all images

In [ ]:
# ============================================================
# SELECT IMAGE(S)
# ============================================================

#The input image should not be much larger than ~2000x3000 pixels, in part to 
#avoid long running times; it is supposed to be a numpy array with 3 channels (RGB). 
#Grains should be well defined in the image and not too small (e.g., only a few pixels in size).

from tkinter import Tk
from tkinter.filedialog import askopenfilenames

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

selected_files = askopenfilenames(parent=root, title="Select foram image(s)", filetypes=[("Image files", "*.jpg *.jpeg *.png *.tif *.tiff *.bmp")])

root.destroy()

image_paths = list(selected_files)

if len(image_paths) == 0:
    raise ValueError("No images were selected.")

print(f"{len(image_paths)} image(s) selected.")

for path in image_paths:
    print(path)

In [ ]:
#Alternative way to load image, use this if the select image above does not work.
# Load your image
#image_paths = "path/to/your/image.jpg"

In [ ]:
# ============================================================
# IMAGE METADATA TABLE
# Add one row per image
# ============================================================

metadata = pd.DataFrame([
    {
        "image_path": path,
        "sample_id": "401_U1611A_63R_CC_0_5",
        "image_id": f"img{i+1:03d}",
        "depth_ccsf": 100,
        "Age_Ma": 0,
        "species": "Orbulina"
    }
    for i, path in enumerate(image_paths)
])

metadata

### For Multiple images, change the index number for each image in current_image function below

In [ ]:
# ============================================================
# LOAD IMAGE AND RUN INITIAL SEGMENTATION
# ============================================================

from PIL import Image
Image.MAX_IMAGE_PIXELS = None

# Select image from metadata table
current_image = metadata.iloc[0]

fname = current_image["image_path"]
print("Processing image:")
print(fname)

# Load image
image = sfi.load_image(fname)

# Automatic box prompts use a whole microscope image.
# Choose "point" for the existing tiled SAM workflow.
sam_prompt_mode = "box"
sam_boxes = np.empty((0, 4), dtype=np.float32)
all_coords = np.empty((0, 2), dtype=int)

if sam_prompt_mode == "box":
    all_grains, image_pred, sam_boxes = sef.predict_with_box_prompts(
        image, unet, sam, min_grain_area=50, min_area=400,
    )
elif sam_prompt_mode == "point":
    all_grains, image_pred, all_coords = sef.predict_large_image(
        fname, unet, sam, use_sam=True, dilation=3, min_area=400.0,
        patch_size=2000, overlap=600, dbs_max_dist=100, remove_edge_grains=False,
    )
else:
    raise ValueError("Choose 'box' or 'point' for sam_prompt_mode.")

# Plot result
fig, ax = plt.subplots(figsize=(14, 10))

sef.plot_image_w_colorful_grains(
    image,
    all_grains,
    ax,
    cmap="tab20b",
    plot_image=True,
    im_alpha=1.0
)

ax.set_title("Automatic segmentation before interaction")
ax.axis("off")

plt.show()

In [ ]:
# ============================================================
# CHECK U-NET PREDICTION AND SAM PROMPTS
# ============================================================

fig, ax = plt.subplots(figsize=(14, 10))

ax.imshow(image)
ax.imshow(image_pred.clip(0, 1), alpha=0.25)

if sam_prompt_mode == "box":
    from matplotlib.patches import Rectangle

    for x0, y0, x1, y1 in sam_boxes:
        ax.add_patch(Rectangle(
            (x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="cyan",
        ))
    ax.set_title(f"U-Net prediction overlay with {len(sam_boxes)} SAM boxes")
elif len(all_coords) > 0:
    ax.scatter(
        all_coords[:, 0],
        all_coords[:, 1],
        c="black",
        s=20
    )
    ax.set_title(f"U-Net prediction overlay with {len(all_coords)} SAM prompt points")
else:
    ax.set_title("U-Net prediction overlay: no SAM prompt points detected")

ax.axis("off")
plt.show()

In [ ]:
# ============================================================
# EXTRACT GRAINS AND MEASURE MORPHOMETRICS
# ============================================================

# Convert segmentation polygons to Grain objects
grains = sfi.polygons_to_grains(all_grains, image=image)

# Measure each detected grain
for grain in tqdm(grains, desc="Measuring detected grains"):
    grain.measure()

print(f"\nMeasured {len(grains)} grains.")

#### Interacting with Segmented Output ####

In [ ]:
# ============================================================
# INTERACTIVE SEGMENTATION CORRECTION
# ============================================================

# Initialize the SAM predictor with the current image
predictor.set_image(image)

# Launch the interactive editor
plot = sfi.ForamPlot(
    grains=grains,
    image=image,
    predictor=predictor,

    # Display settings
    figsize=(12, 8),
    blit=True,
    color_palette="tab20b",

    # Coloring
    color_by=None,

    # Scale information
    scale_m=200*1e-6,     # 500 microm = 0.00005 m
    px_per_m=None        # Will be determined by drawing the scale bar
)

plot.activate()

### Interactive controls (also shown in the figure title bar):

#### Mouse controls:

* Left-click on existing grain: Select/unselect the grain
* Left-click in grain-free area: Place foreground prompt for instant grain creation (auto-create)
#### Alt + Left-click: Place foreground prompt for multi-prompt grain creation (hold Alt, click multiple times, release Alt to create) ####
* Alt + Right-click: Place background prompt for multi-prompt creation
* Shift + Left-drag: Draw a scale bar line (red line) for unit conversion
* Middle-click or Shift + Left-click on grain: Show grain measurement info

#### Keyboard controls:

* d or Delete: Delete selected (highlighted) object
* a/m: add/merge selected foram segments (selected segments must form one connected object)
* z: Undo (delete the most recently created segment/object)
* Ctrl (hold): Temporarily hide all segment masks
* Esc: Remove all prompts and unselect all forams
* c: Create foram segments from existing prompts (alternative to auto-create)

After editing, retrieve the updated grains and deactivate the interactive features:

## Confirm that you have selected all specimens and drawn scale before progressing

In [ ]:
# ============================================================
# RETRIEVE CORRECTED GRAINS FROM INTERACTIVE PLOT
# ============================================================

grains = plot.get_grains()

print(f"Retrieved {len(grains)} corrected grains from ForamPlot.")

In [ ]:
# Turn off interactive features
plot.deactivate()

# Draw the major and minor axes of each grain
plot.draw_axes()

### Saving the corrected mask for further fine tuning

In [ ]:
# ============================================================
# SAVE FINAL CORRECTED MASK
# ============================================================

from pathlib import Path

# Extract final polygons from corrected grains
polygons = [grain.polygon for grain in grains]

# Create labeled image and binary/all-grain mask
labels, mask_all = sef.create_labeled_image(polygons, image)

# Create output folder
output_dir = Path("mask_outputs")
output_dir.mkdir(exist_ok=True)

sample_id = current_image["sample_id"]

mask_path = output_dir / f"{sample_id}_{current_image['species']}_mask.png"

# Save mask
cv2.imwrite(str(mask_path), mask_all.astype(np.uint8))

print(f"Saved corrected mask to: {mask_path}")

### To get the pixels per meters

In [ ]:
# ============================================================
# SUMMARIZE GRAIN MORPHOMETRICS
# ============================================================

# Pixel-to-meter conversion from the interactive scale bar
px_per_m = plot.px_per_m
if px_per_m is None:
    raise ValueError("No scale bar found. Draw the scale bar before running morphometrics.")

print(f"Calibration: {px_per_m:.2f} pixels per meter")

# Compute morphometric summary
summary = sfi.get_summary(grains, px_per_m)

### For repeated segmentation, run cell below to skip measuring scale bar if scale stays the same

In [ ]:
# ============================================================
# PIXEL CALIBRATION
# ============================================================
#228220.37/526894.56/862647.30/360915.04/363020.85/532133.57
PX_PER_M = 357654.28

summary = sfi.get_summary(grains, PX_PER_M)

In [ ]:
foram_data = sef.get_foram_morphometrics(
    summary,
    metadata=current_image
)

foram_data.head()

#### Save the foram data

In [ ]:
foram_data_path, combined_data = sef.save_foram_morphometrics(
    foram_data,
    output_dir="foram_morphometrics_outputs",
)

print(f"Specimen data saved to: {foram_data_path}")
print(f"Total specimens in file: {len(combined_data)}")

#### Export individual specimens

Optional. Run this after saving the morphometrics CSV. Each specimen gets a padded RGB PNG and an aligned binary mask PNG. The manifest links the files to the source image and the existing morphometrics `foram_id`. Crop coordinates use zero-based pixels with exclusive right and bottom bounds.

In [ ]:
from segmenteveryforam.specimen_export import export_specimens

export_dir = export_specimens(image, grains, foram_data, current_image, foram_data_path)
print(f"Exported {len(foram_data)} specimen crops and masks to {export_dir}")

#### To display original image with corresponding foram_id

In [ ]:
# ============================================================
# DISPLAY FINAL GRAINS WITH GRAIN IDs
# ============================================================

from shapely.geometry import Polygon

fig, ax = plt.subplots(figsize=(15, 10))

# Show original image
ax.imshow(image)

for grain, grain_id in zip(grains, foram_data["foram_id"]):

    # Polygon coordinates
    polygon = Polygon(grain.polygon)

    # Centroid
    centroid = polygon.centroid

    # Display grain_id
    ax.text(
        centroid.x,
        centroid.y,
        str(grain_id),
        color="yellow",
        fontsize=8,
        ha="center",
        va="center",
        bbox=dict(facecolor="black", alpha=0.6, edgecolor="none", pad=1)
    )

ax.set_title("Final segmented foraminifera with grain IDs")
ax.axis("off")

plt.show()

### Create average measurement of the foram data 

In [ ]:
sample_summary = sef.summarize_foram_morphometrics(
    combined_data,
    group_by=[
        "sample_id",
        "species",
        "depth_ccsf",
        "Age_Ma",
    ],
)

sample_summary

### Save the average measurement as csv

In [ ]:
summary_path, combined_summary = sef.save_foram_summary(
    sample_summary,
    output_dir="foram_average_measurement_outputs",
)

print(f"Summary saved to: {summary_path}")
print(f"Total samples in summary: {len(combined_summary)}")

## Can Stop here if you don't want to see plots

#### Generate Histogram

In [ ]:
import importlib
import segmenteveryforam.plotting as plotting

importlib.reload(plotting)

import segmenteveryforam as sef
importlib.reload(sef)

In [ ]:
fig, ax = sef.plot_morphometric_histogram(
    foram_data,
    metric="major_diameter_mm",
)

### Boxplots

In [ ]:
fig, ax = sef.plot_morphometric_boxplot(
    combined_data,
    metric="major_diameter_mm",
    vertical="depth_ccsf",
    show_n=True,
)

In [ ]:
fig, ax = sef.plot_morphometric_boxplot(
    combined_data,
    metric="area_mm2",
    vertical="Age_Ma",
    show_n=False,
)

### Generic profile plot

In [ ]:
fig, ax = sef.plot_morphometric_profile(
    sample_summary,
    metric="major_diameter_mm",
    vertical="depth_ccsf",
    statistic="mean",
    uncertainty="sem",
    show_n=True,
)

In [ ]:
fig, ax = sef.plot_morphometric_profile(
    sample_summary,
    metric="major_diameter_mm",
    vertical="depth_ccsf",
    statistic="mean",
    uncertainty="sem",
    show_n=False,
)

In [ ]:
fig, ax = sef.plot_morphometric_profile(
    sample_summary,
    metric="major_diameter_mm",
    vertical="depth_ccsf",
    statistic="p95",
)

In [ ]:
fig, ax = sef.plot_morphometric_profile(
    sample_summary,
    metric="area_mm2",
    vertical="Age_Ma",
    statistic="mean",
    uncertainty="sem",
)